# Loadsmart — Export delivered loads for the latest available month

## Objective

The challenge asks for a CSV containing the `loadsmart_id` values for loads **delivered in the last available month** in the supplied dataset, together with the requested descriptive and financial fields.

> Important interpretation: this requirement is **not** the same as the "last full month" wording used by question Q1. For this export, the challenge explicitly says **last available month**. Therefore, the notebook identifies the latest month represented by `delivery_date` in the modeled data and exports delivered loads from that month.

The notebook is intentionally self-contained: all connection, date logic, SQL, validation, and CSV export happen here.


## 1. Imports and project configuration

We use DuckDB to read the dimensional model and pandas to validate and write the final CSV. Paths are resolved from the repository root so the notebook can be run from the `notebooks/` folder or from the repository root.


In [1]:
from pathlib import Path
import os

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by locating the dbt project directory."""
    start = (start or Path.cwd()).resolve()

    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "dbt_loadsmart" / "dbt_project.yml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Run this notebook from the project or notebooks directory."
    )


PROJECT_ROOT = find_project_root()
DUCKDB_PATH = Path(
    os.environ.get("DUCKDB_PATH", "data/loadsmart.duckdb")
)

if not DUCKDB_PATH.is_absolute():
    DUCKDB_PATH = PROJECT_ROOT / DUCKDB_PATH

OUTPUT_PATH = PROJECT_ROOT / "analysis" / "delivered_loads_latest_available_month.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"DuckDB path  : {DUCKDB_PATH}")
print(f"Output path  : {OUTPUT_PATH}")

if not DUCKDB_PATH.exists():
    raise FileNotFoundError(
        f"DuckDB database not found: {DUCKDB_PATH}. "
        "Run the dbt build first."
    )


Project root : C:\Users\Computador\Desktop\Agent Sandbox\loadsmart-test\loadsmarttechnicalchallenge\loadsmart-challenge
DuckDB path  : C:\Users\Computador\Desktop\Agent Sandbox\loadsmart-test\loadsmarttechnicalchallenge\loadsmart-challenge\data\loadsmart.duckdb
Output path  : C:\Users\Computador\Desktop\Agent Sandbox\loadsmart-test\loadsmarttechnicalchallenge\loadsmart-challenge\analysis\delivered_loads_latest_available_month.csv


## 2. Connect to the dimensional model

The export should be based on the modeled data rather than reading the raw CSV directly. This keeps the notebook aligned with the dimensional model created for the challenge.

The query below uses the analytics fact and dimensions and therefore benefits from the documented relationships already established in dbt.


In [2]:
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)

# Check the available schemas so that failures are easier to diagnose.
schemas = con.execute("SHOW SCHEMAS").fetchdf()
print(schemas)


  database_name     schema_name  current
0     loadsmart            main     True
1     loadsmart  main_analytics    False
2     loadsmart        main_raw    False
3     loadsmart    main_staging    False


## 3. Identify the latest available delivery month

This is the critical distinction from Q1. Q1 asks for the **last full month**. The export requirement instead asks for the **last available month**.

So we calculate the maximum month represented in `delivery_date`, without excluding a partial month. In the supplied dataset, this resolves to **March 2025**.


In [3]:
latest_month_sql = """
SELECT
    date_trunc('month', MAX(delivery_date)) AS latest_available_month,
    MAX(delivery_date) AS latest_delivery_date
FROM main_analytics.fct_loads
WHERE delivery_date IS NOT NULL
"""

latest_month = con.execute(latest_month_sql).fetchone()
latest_available_month = latest_month[0]
latest_delivery_date = latest_month[1]

if latest_available_month is None:
    con.close()
    raise ValueError("No delivery dates are available in fct_loads.")

print(f"Latest available delivery month: {latest_available_month:%Y-%m}")
print(f"Latest delivery date: {latest_delivery_date:%Y-%m-%d}")


Latest available delivery month: 2025-03
Latest delivery date: 2025-03-15


## 4. Check activity in the latest available month

Before exporting, inspect the volume in the latest month. This makes the month-selection logic visible and provides a simple sanity check.

Because the challenge asks specifically for **delivered** loads, the export filters on the model's `is_delivered` flag after identifying the latest month from delivery dates.


In [4]:
monthly_activity_sql = """
SELECT
    date_trunc('month', delivery_date) AS delivery_month,
    COUNT(*) AS all_loads,
    SUM(CASE WHEN is_delivered THEN 1 ELSE 0 END) AS delivered_loads
FROM main_analytics.fct_loads
WHERE delivery_date IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

monthly_activity = con.execute(monthly_activity_sql).fetchdf()
monthly_activity


,delivery_month,all_loads,delivered_loads
0,2024-01-01,278,258.0
1,2024-02-01,297,281.0
2,2024-03-01,420,389.0
3,2024-04-01,463,426.0
4,2024-05-01,478,426.0
5,2024-06-01,555,496.0
6,2024-07-01,334,304.0
7,2024-08-01,454,422.0
8,2024-09-01,483,403.0
9,2024-10-01,503,448.0


## 5. Build the export query

The query below returns exactly the nine fields requested in the challenge:

`loadsmart_id`, `shipper_name`, `delivery_date`, `pickup_city`, `pickup_state`, `delivery_city`, `delivery_state`, `book_price`, and `carrier_name`.

The key filter is: 

```sql
date_trunc('month', f.delivery_date) = latest_available_month
```

There is deliberately **no condition requiring the month to be complete**.


In [5]:
export_sql = """
WITH latest_available AS (
    SELECT
        date_trunc('month', MAX(delivery_date)) AS latest_available_month
    FROM main_analytics.fct_loads
    WHERE delivery_date IS NOT NULL
)
SELECT
    f.loadsmart_id,
    s.shipper_name,
    f.delivery_date,
    l.pickup_city,
    l.pickup_state,
    l.delivery_city,
    l.delivery_state,
    f.book_price,
    c.carrier_name
FROM main_analytics.fct_loads AS f
LEFT JOIN main_analytics.dim_shipper AS s
    ON f.shipper_key = s.shipper_key
LEFT JOIN main_analytics.dim_carrier AS c
    ON f.carrier_key = c.carrier_key
LEFT JOIN main_analytics.dim_lane AS l
    ON f.lane_key = l.lane_key
CROSS JOIN latest_available AS m
WHERE f.is_delivered
  AND date_trunc('month', f.delivery_date) = m.latest_available_month
ORDER BY f.delivery_date, f.loadsmart_id
"""

export_df = con.execute(export_sql).fetchdf()

print(f"Rows to export: {len(export_df):,}")
export_df.head(10)


Rows to export: 1


,loadsmart_id,shipper_name,delivery_date,pickup_city,pickup_state,delivery_city,delivery_state,book_price,carrier_name
0,206665369,Shipper 758,2025-03-15,Lodi,CA,Pacific,WA,1955.56,Carrier 86454


## 6. Validate the result before writing the CSV

These checks make the notebook reproducible and catch common export mistakes:

1. All required columns are present and in the requested order.
2. Every row is delivered.
3. Every row belongs to the latest available month.
4. `loadsmart_id` is unique at the fact grain.
5. At least one row was returned.


In [6]:
required_columns = [
    "loadsmart_id",
    "shipper_name",
    "delivery_date",
    "pickup_city",
    "pickup_state",
    "delivery_city",
    "delivery_state",
    "book_price",
    "carrier_name",
]

assert list(export_df.columns) == required_columns, (
    f"Unexpected columns: {list(export_df.columns)}"
)

assert len(export_df) > 0, "The latest available month returned no delivered loads."

# Re-query the modeled fact table for validation without adding internal
# columns to the final CSV.
validation_sql = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT loadsmart_id) AS distinct_loads,
    SUM(CASE WHEN is_delivered THEN 0 ELSE 1 END) AS non_delivered_rows,
    SUM(
        CASE
            WHEN date_trunc('month', delivery_date) = ? THEN 0
            ELSE 1
        END
    ) AS rows_outside_latest_month
FROM main_analytics.fct_loads
WHERE is_delivered
  AND date_trunc('month', delivery_date) = ?
"""

validation = con.execute(
    validation_sql,
    [latest_available_month, latest_available_month],
).fetchone()

print({
    "row_count": validation[0],
    "distinct_loads": validation[1],
    "non_delivered_rows": validation[2],
    "rows_outside_latest_month": validation[3],
})

assert validation[0] == validation[1], "Duplicate loadsmart_id values detected."
assert validation[2] == 0, "Non-delivered rows detected."
assert validation[3] == 0, "Rows outside the latest available month detected."


{'row_count': 1, 'distinct_loads': 1, 'non_delivered_rows': 0, 'rows_outside_latest_month': 0}


## 7. Preview the final export

A final preview helps verify that the dimensional joins produced the expected descriptive fields before the file is written.


In [7]:
export_df.head(20)


,loadsmart_id,shipper_name,delivery_date,pickup_city,pickup_state,delivery_city,delivery_state,book_price,carrier_name
0,206665369,Shipper 758,2025-03-15,Lodi,CA,Pacific,WA,1955.56,Carrier 86454


## 8. Write the CSV

The final file is written to the `analysis/` directory. `index=False` prevents pandas from adding an extra index column.


In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(export_df):,} delivered loads to:")
print(OUTPUT_PATH)


Exported 1 delivered loads to:
C:\Users\Computador\Desktop\Agent Sandbox\loadsmart-test\loadsmarttechnicalchallenge\loadsmart-challenge\analysis\delivered_loads_latest_available_month.csv


## 9. Final verification

Re-read the generated CSV and verify its shape and columns. This confirms that the file on disk matches the DataFrame that was validated above.


In [9]:
export_check = pd.read_csv(OUTPUT_PATH)

assert list(export_check.columns) == required_columns
assert len(export_check) == len(export_df)

print(f"Final CSV rows   : {len(export_check):,}")
print(f"Final CSV columns: {len(export_check.columns)}")
print(f"Latest month     : {latest_available_month:%Y-%m}")


Final CSV rows   : 1
Final CSV columns: 9
Latest month     : 2025-03


## Conclusion

The export is based on the **latest available delivery month**, not the last full month. For this dataset, the latest available delivery month is **March 2025**. The notebook therefore exports delivered loads with `delivery_date` in March 2025.

This logic is intentionally separate from Q1, where the challenge explicitly asks for the **last full month available**.
